# 02. MMYOLO/COCO JSON에서 class를 읽어 Label Studio bbox UI 적용

이 노트북은 `annotations_mmyolo.json` 같은 COCO/MMYOLO 형식 JSON의 `categories`를 읽고, Label Studio project의 labeling UI를 bbox 작업용으로 세팅합니다.

일반적인 사용 순서:

1. `01_ls_import_images.ipynb`로 project를 만들거나 기존 project id를 확인합니다.
2. `PROJECT_ID`에 Label Studio project id를 입력합니다.
3. `MMYOLO_JSON`에 class 정보를 가져올 JSON 경로를 입력합니다.
4. 먼저 `PREVIEW_ONLY=True`로 XML 미리보기를 확인합니다.
5. 문제가 없으면 `PREVIEW_ONLY=False`로 바꾸고 실행합니다.

주의: `PREVIEW_ONLY=False`는 기존 project의 label config를 실제로 변경합니다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다.")

REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from labelstudio_bbox_tools.config import settings_from_env
from labelstudio_bbox_tools.ui.class_sources import collect_classes_mmyolo
from labelstudio_bbox_tools.ui.label_config import make_label_config_xml, apply_label_config

In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

# Label Studio 브라우저에서 확인한 project id를 넣으세요.
PROJECT_ID = 0

# categories 정보를 읽을 MMYOLO/COCO JSON 경로입니다.
MMYOLO_JSON = Path("/path/to/annotations_mmyolo.json")

# bbox면 RectangleLabels, polygon이면 PolygonLabels로 세팅합니다.
SHAPE_TYPE = "bbox"

# True면 XML 미리보기만 하고 project는 수정하지 않습니다.
PREVIEW_ONLY = True

In [ ]:
classes = collect_classes_mmyolo(MMYOLO_JSON)
print(f"class 개수: {len(classes)}")
print(classes[:20])

xml = make_label_config_xml(classes, shape=SHAPE_TYPE)
print(xml[:2000])

In [ ]:
if PREVIEW_ONLY:
    print("PREVIEW_ONLY=True 이므로 Label Studio project를 수정하지 않습니다.")
else:
    settings = settings_from_env(REPO_ROOT / ".env")
    apply_label_config(
        project_id=PROJECT_ID,
        ls_url=settings.url,
        api_key=settings.api_key,
        classes=classes,
        shape=SHAPE_TYPE,
    )